# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PaNavar369/Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Method of choice : Decision Tree or Logistic Regression

Reason: I will use a simple supervised learning method to compare against my Week-4 rule-based baseline. My baseline prioritizes pages using observable signals such as content staleness and CTR relative to the expected CTR for the page's position tier. The purpose of the model is not to claim that these signals cause ranking movement. Instead, it tests whether a learned model can make useful predictions or classifications on the same decision-support problem.

I prefer a simple interpretable model rather than a complex model because this capstone requires honest comparison and interpretation. A Decision Tree can show which signals and thresholds it relies on, while Logistic Regression can show the directional relationship between features and the predicted outcome.

The model will be compared with my Week-4 baseline using the same data, validation split and evaluation metric.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


I use a time-aware split because the task involves observed search signals that can change over time. Training on earlier observations and evaluating on later observations better represents how the model would be used in practice. The split also helps reduce the risk of future information leaking into the training data.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

df = pd.read_csv(
    "https://raw.githubusercontent.com/PaNavar369/Internship/main/data/raw/content_refresh_anonymized.csv"
)

print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [9]:
print(df.columns.tolist())

features = [
    "days_since_last_update",
    "ctr",
    "avg_position",
    "impressions_90d"
]

target = "trend_direction"
model_df = df[
    ["client_id", "content_id"] + features + [target]
].copy()

model_df = model_df.dropna(subset=features + [target])

print("Rows available for modeling:", len(model_df))
print("Number of clients:", model_df["client_id"].nunique())

model_df.head()


['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
Rows available for modeling: 30000
Number of clients: 32


,client_id,content_id,days_since_last_update,ctr,avg_position,impressions_90d,trend_direction
0,client_f369cb89fc,content_304f48230142,20,0.76,10.6,3803,down
1,client_4e07408562,content_a1fb4e703a9e,25,0.05,20.3,15320,down
2,client_7f2253d7e2,content_9aa793d4d895,20,0.09,36.5,12581,down
3,client_19581e27de,content_331d6c4de07b,22,0.49,6.2,11751,stable
4,client_3fdba35f04,content_d99b7a2d90ca,14,0.13,44.0,19140,down


In [13]:
from sklearn.model_selection import train_test_split

X = model_df[features]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTest class distribution:")
print(y_test.value_counts())

Training rows: 24000
Test rows: 6000

Training class distribution:
trend_direction
down      13010
stable     4770
up         3510
new        1789
flat        921
Name: count, dtype: int64

Test class distribution:
trend_direction
down      3252
stable    1192
up         878
new        447
flat       231
Name: count, dtype: int64


In [14]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print()
print(classification_report(y_test, pred))

Accuracy: 0.6015

              precision    recall  f1-score   support

        down       0.60      0.96      0.74      3252
        flat       0.51      0.31      0.38       231
         new       0.68      0.72      0.70       447
      stable       0.00      0.00      0.00      1192
          up       0.60      0.09      0.16       878

    accuracy                           0.60      6000
   macro avg       0.48      0.42      0.40      6000
weighted avg       0.48      0.60      0.49      6000



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


The model was most useful when the observed signals were clearly separated. Errors were more common when staleness and CTR deficit gave conflicting signals or were close to the thresholds used by the baseline. This suggests that the model may be sensitive to borderline observations rather than providing a clearly superior decision in every case.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


I use AI, data, and structured workflows to turn complex technical problems into practical, evidence-based solutions

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.